# Tablas de la evaluación

Hay una sección por cada pregunta de la evaluación. Son independientes salvo la última, que lee lo que escribe la primera, así que el notebook se corre de arriba abajo. Todas leen las medidas de `../medidas_*.csv` y dejan sus tablas en `salidas/`.

1. Cómo le va a cada configuración en cada tarea
2. Si el oro entra o no al subgrafo
3. Casos en los que el grafo no tiene el oro
4. ¿Por qué la selección empuja hacia el no?
5. Medidas de summary
6. Costo de cada configuración

# Cómo le va a cada configuración en cada tarea

In [1]:
import pandas as pd

from ic import CONFIGURACIONES, SELECCION, comparar, con_respaldo

pd.set_option("display.width", 200)

In [2]:
factoid = pd.read_csv("../medidas_factoid.csv")
list_ = pd.read_csv("../medidas_list.csv")
yesno = pd.read_csv("../medidas_yesno.csv")
summary = pd.read_csv("../medidas_summary.csv")
pd.Series({"factoid": factoid["consulta"].nunique(), "list": list_["consulta"].nunique(), "yes/no": yesno["consulta"].nunique(), "summary": summary["consulta"].nunique()})

factoid    111
list        51
yes/no     100
summary    100
dtype: int64

## factoid

Se mide la exactitud laxa, es decir la pregunta acierta si el oro está entre los cinco primeros candidatos.

Cada línea de la respuesta es un candidato. Se pasa a minúsculas, se le sacan viñetas, paréntesis y la marca [LLM: verify] y se toman las cinco primeras. Un candidato acierta si coincide con algún sinónimo del oro que da BioASQ o si el diccionario de linkear_entidades lo lleva al mismo nodo que el oro. La pregunta vale 1 si acierta alguno de los cinco, y después se promedian las tres repeticiones

In [3]:
por_pregunta = factoid.pivot_table(index="consulta", columns="configuracion", values="laxa")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/medida_principal_factoid.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,111,0.613,NaN,NaN,NaN,0.030,-0.042,0.099
grafo entero,111,0.583,-0.030,-0.099,0.042,NaN,NaN,NaN
solo semillas,111,0.634,0.021,-0.075,0.120,0.051,-0.039,0.141
DIAMOnD,111,0.628,0.015,-0.084,0.114,0.045,-0.039,0.129
DIAMOnD y C3,111,0.613,-0.000,-0.105,0.105,0.030,-0.063,0.120
completa,111,0.622,0.009,-0.093,0.108,0.039,-0.045,0.120


## list

F1 de la lista contra los ítems del oro.

La respuesta se parte en ítems con la misma limpieza que en factoid, y cada ítem se compara con el oro igual que un candidato. La precisión es la fracción de ítems de la respuesta que aciertan algún ítem del oro, y el recall, la fracción de ítems del oro que acierta al menos un ítem de la respuesta. La F1 es la media armónica de las dos. Un ítem del oro con varios sinónimos cuenta una sola vez

In [4]:
por_pregunta = list_.pivot_table(index="consulta", columns="configuracion", values="f1")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/medida_principal_list.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,51,0.426,NaN,NaN,NaN,-0.036,-0.106,0.034
grafo entero,51,0.462,0.036,-0.034,0.106,NaN,NaN,NaN
solo semillas,51,0.536,0.110,0.001,0.222,0.074,-0.016,0.167
DIAMOnD,51,0.552,0.126,0.021,0.232,0.091,-0.003,0.187
DIAMOnD y C3,51,0.539,0.114,0.007,0.221,0.078,-0.016,0.171
completa,51,0.555,0.129,0.020,0.243,0.094,0.011,0.180


In [5]:
list_.groupby("configuracion")[["precision", "recall", "f1"]].mean().reindex(CONFIGURACIONES).round(3)

,precision,recall,f1
configuracion,,,
sin_grafo,0.454,0.445,0.426
grafo_entero,0.461,0.510,0.462
semillas,0.524,0.601,0.536
diamond,0.561,0.594,0.552
diamond_c3,0.539,0.595,0.539
completa,0.555,0.603,0.555


La misma F1 contando solo los sinónimos de BioASQ, sin el diccionario de nodos.

In [6]:
por_pregunta = list_.pivot_table(index="consulta", columns="configuracion", values="f1_sinonimos")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/medida_principal_list_sinonimos.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,51,0.412,NaN,NaN,NaN,-0.021,-0.087,0.044
grafo entero,51,0.432,0.021,-0.044,0.087,NaN,NaN,NaN
solo semillas,51,0.459,0.047,-0.058,0.152,0.027,-0.057,0.113
DIAMOnD,51,0.468,0.056,-0.041,0.154,0.036,-0.047,0.122
DIAMOnD y C3,51,0.454,0.042,-0.059,0.143,0.021,-0.063,0.108
completa,51,0.484,0.073,-0.024,0.173,0.052,-0.022,0.131


Los aciertos que solo cuentan por el diccionario, para leerlos a mano.

In [7]:
aciertos = list_.dropna(subset="solo_diccionario")
aciertos = aciertos.assign(par=aciertos["solo_diccionario"].str.split(r" \| ")).explode("par")
por_configuracion = aciertos.groupby("par")["configuracion"].unique().str.join(", ").rename("configuraciones").reset_index()
por_configuracion["legitimo"] = ""
por_configuracion.to_csv("salidas/aciertos_solo_diccionario.csv", index=False)
len(por_configuracion)

21

## yes/no



In [8]:
por_pregunta = yesno.pivot_table(index="consulta", columns="configuracion", values="acierto")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/medida_principal_yesno.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,100,0.847,NaN,NaN,NaN,0.030,-0.017,0.080
grafo entero,100,0.817,-0.030,-0.080,0.017,NaN,NaN,NaN
solo semillas,100,0.737,-0.110,-0.190,-0.030,-0.080,-0.170,0.010
DIAMOnD,100,0.693,-0.153,-0.230,-0.077,-0.123,-0.210,-0.037
DIAMOnD y C3,100,0.723,-0.123,-0.207,-0.040,-0.093,-0.190,0.003
completa,100,0.763,-0.083,-0.167,-0.003,-0.053,-0.137,0.030


In [9]:
round((yesno.groupby("consulta")["oro"].first() == "yes").mean(), 3)

np.float64(0.78)

In [10]:
pd.read_csv("../f1_macro_yesno.csv", index_col="configuracion").round(3)

,f1_macro
configuracion,
sin_grafo,0.774
grafo_entero,0.734
semillas,0.702
diamond,0.655
diamond_c3,0.689
completa,0.706


## summary

Cobertura de RAGChecker: la fracción de afirmaciones de la primera ideal que la respuesta implica.

El juez es gpt-oss:20b, de otra familia que el modelo que responde. Parte la primera respuesta ideal en afirmaciones cortas, una sola vez por pregunta, y para cada una decide si la respuesta la implica, la contradice o no dice nada de ella. Solo cuentan las implicadas, así que lo que la respuesta agrega y la ideal no menciona no le resta.

In [11]:
por_pregunta = summary.pivot_table(index="consulta", columns="configuracion", values="rc_cobertura")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/medida_principal_summary.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,100,0.488,NaN,NaN,NaN,0.042,0.013,0.072
grafo entero,100,0.446,-0.042,-0.072,-0.013,NaN,NaN,NaN
solo semillas,100,0.342,-0.147,-0.193,-0.103,-0.104,-0.141,-0.070
DIAMOnD,100,0.361,-0.127,-0.170,-0.084,-0.085,-0.121,-0.049
DIAMOnD y C3,100,0.361,-0.128,-0.168,-0.088,-0.085,-0.119,-0.054
completa,100,0.347,-0.142,-0.182,-0.102,-0.099,-0.135,-0.066


## Si el subgrafo vacío respondiera

Un subgrafo vacío se abstiene y vale cero. Acá esas preguntas valen lo que el modelo solo, que es lo que haría un sistema real.

Preguntas con subgrafo vacío por configuración.

In [12]:
pd.DataFrame({"factoid": factoid.groupby("configuracion")["abstiene"].mean(), "list": list_.groupby("configuracion")["abstiene"].mean(),
              "yes/no": yesno.groupby("configuracion")["abstiene"].mean(), "summary": summary.groupby("configuracion")["abstiene"].mean()}).reindex(SELECCION).mul([111, 51, 100, 100]).round(0)

,factoid,list,yes/no,summary
configuracion,,,,
semillas,6.0,3.0,4.0,5.0
diamond,6.0,3.0,4.0,5.0
diamond_c3,6.0,3.0,4.0,5.0
completa,0.0,0.0,1.0,0.0


In [13]:
tabla = comparar(con_respaldo(factoid, "laxa"))
tabla.to_csv("salidas/respaldo_factoid.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,111,0.613,NaN,NaN,NaN,0.030,-0.042,0.099
grafo entero,111,0.583,-0.030,-0.099,0.042,NaN,NaN,NaN
solo semillas,111,0.679,0.066,-0.021,0.156,0.096,0.012,0.180
DIAMOnD,111,0.673,0.060,-0.030,0.150,0.090,0.012,0.171
DIAMOnD y C3,111,0.658,0.045,-0.051,0.144,0.075,-0.012,0.162
completa,111,0.622,0.009,-0.093,0.108,0.039,-0.045,0.120


In [14]:
tabla = comparar(con_respaldo(list_, "f1"))
tabla.to_csv("salidas/respaldo_list.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,51,0.426,NaN,NaN,NaN,-0.036,-0.106,0.034
grafo entero,51,0.462,0.036,-0.034,0.106,NaN,NaN,NaN
solo semillas,51,0.547,0.121,0.014,0.231,0.086,-0.000,0.175
DIAMOnD,51,0.563,0.138,0.036,0.242,0.102,0.014,0.194
DIAMOnD y C3,51,0.550,0.125,0.023,0.230,0.089,0.001,0.179
completa,51,0.555,0.129,0.020,0.243,0.094,0.011,0.180


In [15]:
tabla = comparar(con_respaldo(yesno, "acierto"))
tabla.to_csv("salidas/respaldo_yesno.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,100,0.847,NaN,NaN,NaN,0.030,-0.017,0.080
grafo entero,100,0.817,-0.030,-0.080,0.017,NaN,NaN,NaN
solo semillas,100,0.770,-0.077,-0.153,-0.003,-0.047,-0.130,0.037
DIAMOnD,100,0.727,-0.120,-0.193,-0.047,-0.090,-0.170,-0.010
DIAMOnD y C3,100,0.757,-0.090,-0.170,-0.013,-0.060,-0.147,0.030
completa,100,0.773,-0.073,-0.153,0.007,-0.043,-0.123,0.037


In [16]:
tabla = comparar(con_respaldo(summary, "rc_cobertura"))
tabla.to_csv("salidas/respaldo_summary.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,100,0.488,NaN,NaN,NaN,0.042,0.013,0.072
grafo entero,100,0.446,-0.042,-0.072,-0.013,NaN,NaN,NaN
solo semillas,100,0.359,-0.129,-0.171,-0.089,-0.087,-0.124,-0.052
DIAMOnD,100,0.379,-0.109,-0.150,-0.070,-0.067,-0.104,-0.032
DIAMOnD y C3,100,0.378,-0.110,-0.148,-0.074,-0.068,-0.101,-0.036
completa,100,0.347,-0.142,-0.182,-0.102,-0.099,-0.135,-0.066


# Ver si el oro entra o no al subgrafo

El oro es la respuesta de referencia de BioASQ, y la respuesta es lo que genera el sistema. Se comparan seis configuraciones: el modelo solo, el grafo entero (GraphRAG sin selección) y las cuatro selecciones (solo semillas, DIAMOnD, DIAMOnD y C3, y completa). El oro entra si alguno de sus nodos está en el subgrafo. En list se mira la cobertura, que es la fracción de ítems del oro que entraron.

In [17]:
import pandas as pd

from ic import CONFIGURACIONES, SELECCION, NOMBRES, comparar, con_respaldo, interaccion

pd.set_option("display.width", 200)

In [18]:
factoid = pd.read_csv("../medidas_factoid.csv")
list_ = pd.read_csv("../medidas_list.csv")
adentro = pd.read_csv("../cobertura_factoid.csv", index_col="consulta")
cobertura = pd.read_csv("../cobertura_list.csv", index_col="consulta")
pd.Series({"factoid": len(adentro), "list": len(cobertura)})

factoid    111
list        51
dtype: int64

## Sin modelo

Preguntas de factoid con el oro en el subgrafo.

In [19]:
pd.DataFrame({"preguntas": adentro.sum(), "fraccion": adentro.mean().round(3)})

,preguntas,fraccion
semillas,53,0.477
diamond,53,0.477
diamond_c3,53,0.477
completa,57,0.514


Ítems de list en el subgrafo, sobre el total.

In [20]:
items = cobertura["items_oro"].sum()
pd.DataFrame({"items": [cobertura[f"cubiertos_{c}"].sum() for c in SELECCION], "total": items}, index=SELECCION)

,items,total
semillas,76,177
diamond,77,177
diamond_c3,78,177
completa,86,177


## factoid

Laxa. Los grupos se fijan con el subgrafo de completa, que contiene a los otros tres, y son los mismos para las seis configuraciones.

In [21]:
por_pregunta = factoid.pivot_table(index="consulta", columns="configuracion", values="laxa")[CONFIGURACIONES]

In [22]:
tabla = comparar(por_pregunta.loc[adentro.index[adentro["completa"]]])
tabla.to_csv("salidas/factoid_adentro.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,57,0.673,NaN,NaN,NaN,0.006,-0.064,0.082
grafo entero,57,0.667,-0.006,-0.082,0.064,NaN,NaN,NaN
solo semillas,57,0.842,0.170,0.047,0.298,0.175,0.058,0.292
DIAMOnD,57,0.842,0.170,0.041,0.298,0.175,0.058,0.304
DIAMOnD y C3,57,0.860,0.187,0.053,0.327,0.193,0.064,0.327
completa,57,0.830,0.158,0.023,0.292,0.164,0.041,0.287


El oro entró a los cuatro subgrafos, es decir, sin depender del RWR.

In [23]:
tabla = comparar(por_pregunta.loc[adentro.index[adentro[SELECCION].all(axis=1)]])
tabla.to_csv("salidas/factoid_adentro_de_los_cuatro.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,53,0.667,NaN,NaN,NaN,0.006,-0.075,0.088
grafo entero,53,0.660,-0.006,-0.088,0.075,NaN,NaN,NaN
solo semillas,53,0.887,0.220,0.101,0.340,0.226,0.119,0.340
DIAMOnD,53,0.887,0.220,0.101,0.346,0.226,0.113,0.346
DIAMOnD y C3,53,0.906,0.239,0.107,0.377,0.245,0.126,0.371
completa,53,0.836,0.170,0.031,0.314,0.176,0.044,0.308


El oro no entró.

In [24]:
tabla = comparar(por_pregunta.loc[adentro.index[~adentro["completa"]]])
tabla.to_csv("salidas/factoid_afuera.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,54,0.549,NaN,NaN,NaN,0.056,-0.068,0.179
grafo entero,54,0.494,-0.056,-0.179,0.068,NaN,NaN,NaN
solo semillas,54,0.414,-0.136,-0.272,0.000,-0.080,-0.204,0.037
DIAMOnD,54,0.401,-0.148,-0.284,-0.012,-0.093,-0.198,0.006
DIAMOnD y C3,54,0.352,-0.198,-0.333,-0.056,-0.142,-0.253,-0.037
completa,54,0.401,-0.148,-0.284,-0.012,-0.093,-0.191,0.006


Cuánto cambia la diferencia contra el modelo solo entre adentro y afuera. El grafo entero no depende del subgrafo y sirve de control.

In [25]:
tabla = interaccion(por_pregunta.loc[adentro.index[adentro["completa"]]], por_pregunta.loc[adentro.index[~adentro["completa"]]])
tabla.to_csv("salidas/factoid_interaccion.csv")
tabla

,interaccion,desde,hasta
configuracion,,,
grafo entero,0.050,-0.096,0.192
solo semillas,0.305,0.118,0.494
DIAMOnD,0.318,0.131,0.509
DIAMOnD y C3,0.385,0.191,0.583
completa,0.306,0.114,0.499


## list

F1, en dos grupos: hasta la mitad de los ítems adentro y más de la mitad. Las listas de oro son cortas, y con un tercer grupo quedarían muy pocas preguntas en cada uno.

In [26]:
por_pregunta = list_.pivot_table(index="consulta", columns="configuracion", values="f1")[CONFIGURACIONES]
tabla = comparar(por_pregunta.loc[cobertura.index[cobertura["grupo_completa"] == "mas_de_la_mitad"]])
tabla.to_csv("salidas/list_mas_de_la_mitad.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,21,0.536,NaN,NaN,NaN,-0.019,-0.119,0.080
grafo entero,21,0.555,0.019,-0.080,0.119,NaN,NaN,NaN
solo semillas,21,0.714,0.178,0.002,0.353,0.158,0.019,0.314
DIAMOnD,21,0.734,0.198,0.025,0.372,0.179,0.024,0.344
DIAMOnD y C3,21,0.703,0.167,-0.002,0.341,0.148,0.000,0.309
completa,21,0.665,0.130,-0.043,0.309,0.110,-0.031,0.268


In [27]:
tabla = comparar(por_pregunta.loc[cobertura.index[cobertura["grupo_completa"] == "hasta_la_mitad"]])
tabla.to_csv("salidas/list_hasta_la_mitad.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,30,0.349,NaN,NaN,NaN,-0.047,-0.145,0.049
grafo entero,30,0.396,0.047,-0.049,0.145,NaN,NaN,NaN
solo semillas,30,0.412,0.063,-0.074,0.202,0.016,-0.099,0.128
DIAMOnD,30,0.425,0.076,-0.049,0.207,0.029,-0.082,0.138
DIAMOnD y C3,30,0.425,0.076,-0.058,0.213,0.029,-0.089,0.141
completa,30,0.478,0.129,-0.005,0.276,0.082,-0.013,0.184


La misma interacción para list.

In [28]:
tabla = interaccion(por_pregunta.loc[cobertura.index[cobertura["grupo_completa"] == "mas_de_la_mitad"]],
                    por_pregunta.loc[cobertura.index[cobertura["grupo_completa"] == "hasta_la_mitad"]])
tabla.to_csv("salidas/list_interaccion.csv")
tabla

,interaccion,desde,hasta
configuracion,,,
grafo entero,-0.028,-0.168,0.112
solo semillas,0.115,-0.110,0.336
DIAMOnD,0.122,-0.096,0.336
DIAMOnD y C3,0.091,-0.127,0.308
completa,0.000,-0.229,0.224


Precisión, recall y F1 por grupo.

In [29]:
grupos = list_["consulta"].map(cobertura["grupo_completa"]).rename("grupo")
metricas = list_.groupby([grupos, "configuracion"])[["precision", "recall", "f1"]].mean().round(3)
metricas = metricas.reindex(CONFIGURACIONES, level="configuracion")
metricas.to_csv("salidas/list_metricas_por_grupo.csv")
metricas

precision  recall     f1
grupo           configuracion                          
hasta_la_mitad  sin_grafo          0.376   0.351  0.349
                grafo_entero       0.375   0.450  0.396
                semillas           0.416   0.427  0.412
                diamond            0.447   0.435  0.425
                diamond_c3         0.443   0.435  0.425
                completa           0.491   0.487  0.478
mas_de_la_mitad sin_grafo          0.567   0.579  0.536
                grafo_entero       0.584   0.595  0.555
                semillas           0.679   0.850  0.714
                diamond            0.725   0.823  0.734
                diamond_c3         0.675   0.823  0.703
                completa           0.647   0.770  0.665

Ahora cada selección con los grupos de su propio subgrafo. Los grupos cambian entre filas, así que el modelo solo y el grafo entero se comparan sobre las preguntas de cada una.

In [30]:
filas = []
for configuracion in SELECCION:
    for grupo in ["mas_de_la_mitad", "hasta_la_mitad"]:
        preguntas = cobertura.index[cobertura[f"grupo_{configuracion}"] == grupo]
        fila = comparar(por_pregunta.loc[preguntas]).loc[NOMBRES[configuracion]]
        fila["grupo"] = grupo
        filas.append(fila)
tabla = pd.DataFrame(filas).astype({"preguntas": int})
tabla.to_csv("salidas/list_por_subgrafo_propio.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta,grupo
solo semillas,20,0.743,0.220,0.058,0.386,0.183,0.040,0.335,mas_de_la_mitad
solo semillas,31,0.403,0.039,-0.099,0.182,0.005,-0.108,0.116,hasta_la_mitad
DIAMOnD,20,0.764,0.242,0.084,0.401,0.204,0.050,0.364,mas_de_la_mitad
DIAMOnD,31,0.415,0.052,-0.078,0.186,0.017,-0.090,0.126,hasta_la_mitad
DIAMOnD y C3,21,0.703,0.167,-0.002,0.341,0.148,0.000,0.309,mas_de_la_mitad
DIAMOnD y C3,30,0.425,0.076,-0.058,0.213,0.029,-0.089,0.141,hasta_la_mitad
completa,21,0.665,0.130,-0.043,0.309,0.110,-0.031,0.268,mas_de_la_mitad
completa,30,0.478,0.129,-0.005,0.276,0.082,-0.013,0.184,hasta_la_mitad


## Controles de factoid

Con el oro afuera, las preguntas de subgrafo vacío valen lo que vale el modelo solo.

In [31]:
tabla = comparar(con_respaldo(factoid, "laxa").loc[adentro.index[~adentro["completa"]]])
tabla.to_csv("salidas/factoid_afuera_respaldo.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,54,0.549,NaN,NaN,NaN,0.056,-0.068,0.179
grafo entero,54,0.494,-0.056,-0.179,0.068,NaN,NaN,NaN
solo semillas,54,0.488,-0.062,-0.185,0.062,-0.006,-0.130,0.111
DIAMOnD,54,0.475,-0.074,-0.191,0.043,-0.019,-0.123,0.080
DIAMOnD y C3,54,0.426,-0.123,-0.247,0.000,-0.068,-0.179,0.037
completa,54,0.401,-0.148,-0.284,-0.012,-0.093,-0.191,0.006


Con el oro adentro, contando solo los sinónimos de BioASQ.

In [32]:
sinonimos = factoid.pivot_table(index="consulta", columns="configuracion", values="laxa_sinonimos")[CONFIGURACIONES]
tabla = comparar(sinonimos.loc[adentro.index[adentro["completa"]]])
tabla.to_csv("salidas/factoid_adentro_sinonimos.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,57,0.538,NaN,NaN,NaN,-0.018,-0.094,0.058
grafo entero,57,0.556,0.018,-0.058,0.094,NaN,NaN,NaN
solo semillas,57,0.632,0.094,-0.041,0.228,0.076,-0.035,0.187
DIAMOnD,57,0.591,0.053,-0.088,0.199,0.035,-0.088,0.164
DIAMOnD y C3,57,0.626,0.088,-0.058,0.234,0.070,-0.058,0.199
completa,57,0.637,0.099,-0.035,0.240,0.082,-0.035,0.199


La misma interacción contando solo los sinónimos.

In [33]:
tabla = interaccion(sinonimos.loc[adentro.index[adentro["completa"]]], sinonimos.loc[adentro.index[~adentro["completa"]]])
tabla.to_csv("salidas/factoid_interaccion_sinonimos.csv")
tabla

,interaccion,desde,hasta
configuracion,,,
grafo entero,0.030,-0.108,0.169
solo semillas,0.199,0.008,0.394
DIAMOnD,0.164,-0.024,0.356
DIAMOnD y C3,0.230,0.032,0.436
completa,0.217,0.026,0.412


# Casos en los que el grafo no tiene el oro


In [34]:
import json

import pandas as pd

from ic import CONFIGURACIONES, SELECCION, comparar

pd.set_option("display.width", 200)

In [35]:
por_oro = pd.read_csv("../medidas_list_por_oro.csv")
list_ = pd.read_csv("../medidas_list.csv")
factoid = pd.read_csv("../medidas_factoid.csv")
sin_oro = pd.read_csv("../medidas_factoid_sin_oro.csv")
with open("../preguntas/muestra_list_por_oro.json") as archivo:
    muestra = json.load(archivo)
fraccion = pd.Series({consulta: datos["fraccion_oro"] for consulta, datos in muestra.items()})
pd.Series({"nada": (fraccion == 0).sum(), "menos de la mitad": ((fraccion > 0) & (fraccion < 0.5)).sum(),
           "la mitad o más, sin llegar a todo": ((fraccion >= 0.5) & (fraccion < 1)).sum(), "todo": (fraccion == 1).sum()})

nada                                 25
menos de la mitad                    25
la mitad o más, sin llegar a todo    18
todo                                  7
dtype: int64

## list según cuánto del oro tiene nodo



In [36]:
por_pregunta = por_oro.pivot_table(index="consulta", columns="configuracion", values="f1_sinonimos")[CONFIGURACIONES]
fraccion = fraccion.reindex(por_pregunta.index)

Ningún ítem con nodo:

In [37]:
tabla = comparar(por_pregunta[fraccion == 0])
tabla.to_csv("salidas/por_oro_nada.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,25,0.061,NaN,NaN,NaN,-0.053,-0.111,-0.012
grafo entero,25,0.115,0.053,0.012,0.111,NaN,NaN,NaN
solo semillas,25,0.083,0.022,-0.024,0.076,-0.032,-0.080,-0.001
DIAMOnD,25,0.091,0.029,-0.047,0.113,-0.024,-0.095,0.053
DIAMOnD y C3,25,0.115,0.053,-0.012,0.132,-0.000,-0.053,0.068
completa,25,0.089,0.028,-0.048,0.110,-0.026,-0.096,0.052


Menos de la mitad de los ítems con nodo:

In [38]:
tabla = comparar(por_pregunta[(fraccion > 0) & (fraccion < 0.5)])
tabla.to_csv("salidas/por_oro_menos_de_la_mitad.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,25,0.102,NaN,NaN,NaN,-0.015,-0.052,0.020
grafo entero,25,0.117,0.015,-0.020,0.052,NaN,NaN,NaN
solo semillas,25,0.096,-0.006,-0.048,0.031,-0.021,-0.053,0.009
DIAMOnD,25,0.086,-0.016,-0.067,0.036,-0.031,-0.070,0.005
DIAMOnD y C3,25,0.105,0.003,-0.043,0.047,-0.012,-0.050,0.024
completa,25,0.087,-0.016,-0.062,0.029,-0.030,-0.068,0.004


La mitad de los ítems con nodo o más, pero sin llegar a todos:

In [39]:
tabla = comparar(por_pregunta[(fraccion >= 0.5) & (fraccion < 1)])
tabla.to_csv("salidas/por_oro_la_mitad_sin_todo.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,18,0.113,NaN,NaN,NaN,-0.061,-0.133,0.005
grafo entero,18,0.174,0.061,-0.005,0.133,NaN,NaN,NaN
solo semillas,18,0.250,0.137,0.013,0.275,0.076,-0.017,0.191
DIAMOnD,18,0.222,0.109,-0.005,0.237,0.048,-0.033,0.147
DIAMOnD y C3,18,0.211,0.098,-0.035,0.233,0.037,-0.059,0.144
completa,18,0.246,0.133,-0.015,0.292,0.072,-0.034,0.197


Todos los ítems con nodo, que es la muestra entera de list.

In [40]:
por_pregunta = list_.pivot_table(index="consulta", columns="configuracion", values="f1_sinonimos")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/por_oro_todo.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,51,0.412,NaN,NaN,NaN,-0.021,-0.087,0.044
grafo entero,51,0.432,0.021,-0.044,0.087,NaN,NaN,NaN
solo semillas,51,0.459,0.047,-0.058,0.152,0.027,-0.057,0.113
DIAMOnD,51,0.468,0.056,-0.041,0.154,0.036,-0.047,0.122
DIAMOnD y C3,51,0.454,0.042,-0.059,0.143,0.021,-0.063,0.108
completa,51,0.484,0.073,-0.024,0.173,0.052,-0.022,0.131


Media por pregunta de la fracción de ítems del oro que entró a cada subgrafo. Acá la mitad o más incluye las 7 preguntas con todos los ítems, y el total de ítems incluye los que no tienen nodo.

In [41]:
pd.read_csv("../por_oro_list.csv").pivot_table(index="grupo_oro", columns="configuracion", values="oro_en_el_subgrafo")[SELECCION].round(3)

configuracion,semillas,diamond,diamond_c3,completa
grupo_oro,,,,
la_mitad_o_mas,0.193,0.193,0.193,0.213
menos_de_la_mitad,0.040,0.040,0.040,0.044
nada,0.000,0.000,0.000,0.000


## factoid con nodo y sin nodo

Laxa contando solo los sinónimos de BioASQ. Es descriptivo: los dos conjuntos tienen distinto tipo de pregunta, y sin nodo el oro suele ser una frase. Por eso esos valores son un piso. "TAZ" cuenta como error contra "Tafazzin (TAZ) gene".

In [42]:
por_pregunta = factoid.pivot_table(index="consulta", columns="configuracion", values="laxa_sinonimos")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/factoid_con_nodo.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,111,0.477,NaN,NaN,NaN,-0.003,-0.072,0.063
grafo entero,111,0.480,0.003,-0.063,0.072,NaN,NaN,NaN
solo semillas,111,0.474,-0.003,-0.099,0.096,-0.006,-0.090,0.078
DIAMOnD,111,0.450,-0.027,-0.123,0.069,-0.030,-0.114,0.054
DIAMOnD y C3,111,0.453,-0.024,-0.126,0.078,-0.027,-0.114,0.060
completa,111,0.471,-0.006,-0.102,0.093,-0.009,-0.087,0.069


In [43]:
por_pregunta = sin_oro.pivot_table(index="consulta", columns="configuracion", values="laxa_sinonimos")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/factoid_sin_nodo.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,111,0.084,NaN,NaN,NaN,-0.027,-0.060,-0.003
grafo entero,111,0.111,0.027,0.003,0.060,NaN,NaN,NaN
solo semillas,111,0.105,0.021,-0.027,0.072,-0.006,-0.045,0.033
DIAMOnD,111,0.093,0.009,-0.036,0.057,-0.018,-0.057,0.021
DIAMOnD y C3,111,0.096,0.012,-0.036,0.060,-0.015,-0.057,0.027
completa,111,0.102,0.018,-0.018,0.057,-0.009,-0.036,0.018


# la selección empuja hacia el no?


In [44]:
import pandas as pd

from ic import CONFIGURACIONES, comparar

pd.set_option("display.width", 200)

In [45]:
yesno = pd.read_csv("../medidas_yesno.csv")
origen = pd.read_csv("../origen_yesno.csv")
yesno["consulta"].nunique()

100

Cuántas veces dice yes y cuántas no cada configuración. Un subgrafo vacío se abstiene y cuenta como error.

In [46]:
pd.crosstab([yesno["oro"], yesno["dada"].fillna("abstiene (subgrafo vacio)")], yesno["configuracion"])[CONFIGURACIONES]

configuracion                  sin_grafo  grafo_entero  semillas  diamond  diamond_c3  completa
oro dada                                                                                       
no  no                                42            39        48       45          48        46
    yes                               24            27        18       21          18        20
yes abstiene (subgrafo vacio)          0             0        12       12          12         3
    no                                22            28        49       59          53        48
    yes                              212           206       173      163         169       183

## Según el map

Si el map no puntúa ningún reporte, el reduce no recibe nada del grafo. Fracción de respuestas con reportes puntuados y exactitud en cada caso. Los subgrafos vacíos no entran.

In [47]:
por_map = origen.groupby(["configuracion", "map_con_puntos"])["acierto"].mean().unstack().round(3)
por_map["fraccion_con_puntos"] = origen.groupby("configuracion")["map_con_puntos"].mean().round(3)
por_map.reindex(CONFIGURACIONES[1:])

map_con_puntos,False,True,fraccion_con_puntos
configuracion,,,
grafo_entero,0.841,0.750,0.267
semillas,0.889,0.720,0.719
diamond,0.815,0.679,0.681
diamond_c3,0.839,0.716,0.698
completa,0.822,0.749,0.697


Las mismas respuestas contra el modelo solo en la misma pregunta y repetición.

In [48]:
solo = yesno.loc[yesno["configuracion"] == "sin_grafo", ["consulta", "rep", "acierto"]].rename(columns={"acierto": "modelo_solo"})
apareado = origen.merge(solo, on=["consulta", "rep"])
apareado.groupby(["configuracion", "map_con_puntos"])[["acierto", "modelo_solo"]].mean().round(3)

acierto  modelo_solo
configuracion map_con_puntos                      
completa      False             0.822        0.767
              True              0.749        0.879
diamond       False             0.815        0.772
              True              0.679        0.883
diamond_c3    False             0.839        0.816
              True              0.716        0.861
grafo_entero  False             0.841        0.841
              True              0.750        0.862
semillas      False             0.889        0.827
              True              0.720        0.855

# Medidas de summary

In [49]:
import json

import pandas as pd

from ic import CONFIGURACIONES, SELECCION, comparar

pd.set_option("display.width", 200)

In [50]:
summary = pd.read_csv("../medidas_summary.csv")
summary["consulta"].nunique()

100

ROUGE-SU4 cuenta las palabras y los pares de palabras que la respuesta comparte con la mejor ideal.

In [51]:
por_pregunta = summary.pivot_table(index="consulta", columns="configuracion", values="rouge_su4")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/summary_rouge.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,100,0.109,NaN,NaN,NaN,-0.004,-0.007,-0.001
grafo entero,100,0.113,0.004,0.001,0.007,NaN,NaN,NaN
solo semillas,100,0.098,-0.011,-0.019,-0.005,-0.015,-0.022,-0.009
DIAMOnD,100,0.095,-0.014,-0.022,-0.007,-0.018,-0.025,-0.012
DIAMOnD y C3,100,0.099,-0.010,-0.018,-0.003,-0.014,-0.021,-0.008
completa,100,0.101,-0.008,-0.014,-0.003,-0.012,-0.017,-0.007


BERTScore mide el parecido entre los embeddings de la respuesta y los de la mejor ideal.

Cada token de la respuesta se representa con su embedding contextual de roberta-large y se empareja con el token más parecido de la ideal, y lo mismo al revés.

In [52]:
por_pregunta = summary.pivot_table(index="consulta", columns="configuracion", values="bertscore")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/summary_bertscore.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,100,0.195,NaN,NaN,NaN,-0.011,-0.019,-0.002
grafo entero,100,0.206,0.011,0.002,0.019,NaN,NaN,NaN
solo semillas,100,0.162,-0.033,-0.049,-0.017,-0.044,-0.059,-0.029
DIAMOnD,100,0.155,-0.040,-0.058,-0.022,-0.051,-0.068,-0.034
DIAMOnD y C3,100,0.158,-0.037,-0.055,-0.020,-0.048,-0.064,-0.032
completa,100,0.159,-0.036,-0.053,-0.020,-0.047,-0.063,-0.031


Cobertura de RAGChecker: la fracción de afirmaciones de la primera ideal que la respuesta implica.

In [53]:
por_pregunta = summary.pivot_table(index="consulta", columns="configuracion", values="rc_cobertura")[CONFIGURACIONES]
tabla = comparar(por_pregunta)
tabla.to_csv("salidas/summary_ragchecker.csv")
tabla

,preguntas,media,menos solo,solo desde,solo hasta,menos entero,entero desde,entero hasta
configuracion,,,,,,,,
modelo solo,100,0.488,NaN,NaN,NaN,0.042,0.013,0.072
grafo entero,100,0.446,-0.042,-0.072,-0.013,NaN,NaN,NaN
solo semillas,100,0.342,-0.147,-0.193,-0.103,-0.104,-0.141,-0.070
DIAMOnD,100,0.361,-0.127,-0.170,-0.084,-0.085,-0.121,-0.049
DIAMOnD y C3,100,0.361,-0.128,-0.168,-0.088,-0.085,-0.119,-0.054
completa,100,0.347,-0.142,-0.182,-0.102,-0.099,-0.135,-0.066


Un ejemplo con una pregunta. Cada fila es una afirmación que el juez sacó de la primera ideal, escrita como tripla (sujeto, predicado, objeto), y cada columna dice qué decidió el juez para la respuesta de esa configuración. Entailment quiere decir que la respuesta implica la afirmación, y es lo único que suma.

In [54]:
with open("../summary/ragchecker.json") as archivo:
    juicios = pd.DataFrame(json.load(archivo)["results"])
juicios[["rep", "consulta", "configuracion"]] = juicios["query_id"].str.split("/", expand=True)
#el ejemplo necesita la pregunta, la ideal y las afirmaciones, que el ragchecker.json publicado no trae: hay que volver
#a correr el juez (../medir_summary_ragchecker.py)
con_texto = "gt_answer_claims" in juicios
if con_texto:
    ejemplo = juicios[(juicios["consulta"] == "apalutamide_000046") & (juicios["rep"] == "rep1")].set_index("configuracion")
    print(ejemplo.loc["semillas", "query"])
    print(ejemplo.loc["semillas", "gt_answer"])

Describe the mechanism of action of apalutamide.
Apalutamide is a second-generation antiandrogen that inhibits the binding of androgen to androgen receptor (AR), nuclear translocation of the androgen-AR complex, and binding of AR transcription complex to DNA-binding sites and transcription elements. It does not show antagonist-to-agonist switch like bicalutamide. It is emerging as additional new option to treat castration-resistant prostate cancer.


In [55]:
#sin cortar el texto de las afirmaciones
pd.set_option("display.max_colwidth", None)
if con_texto:
    afirmaciones = [" | ".join(afirmacion) for afirmacion in ejemplo.loc["semillas", "gt_answer_claims"]]
    etiquetas = pd.DataFrame(ejemplo["response2answer"].to_dict(), index=afirmaciones)[CONFIGURACIONES]
    display(etiquetas)

,sin_grafo,grafo_entero,semillas,diamond,diamond_c3,completa
Apalutamide | is a | second-generation antiandrogen,Entailment,Neutral,Neutral,Neutral,Neutral,Neutral
Apalutamide | inhibits | binding of androgen to androgen receptor (AR),Entailment,Entailment,Entailment,Entailment,Entailment,Neutral
Apalutamide | inhibits | nuclear translocation of the androgen-AR complex,Entailment,Entailment,Entailment,Entailment,Entailment,Entailment
Apalutamide | inhibits | binding of AR transcription complex to DNA-binding sites and transcription elements,Entailment,Entailment,Neutral,Entailment,Entailment,Neutral
Apalutamide | does not show | antagonist-to-agonist switch,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral
Apalutamide | is emerging as | new option to treat castration-resistant prostate cancer,Entailment,Entailment,Entailment,Entailment,Entailment,Neutral


La cobertura de cada respuesta es la fracción de filas con Entailment.

In [56]:
if con_texto:
    display((etiquetas == "Entailment").mean().round(2))

sin_grafo       0.83
grafo_entero    0.67
semillas        0.50
diamond         0.67
diamond_c3      0.67
completa        0.17
dtype: float64

## Influye el largo de la respuesta?

Mediana de palabras por respuesta:

In [57]:
summary[~summary["abstiene"]].groupby("configuracion")["palabras"].median().reindex(CONFIGURACIONES)

configuracion
sin_grafo       173.0
grafo_entero    143.0
semillas        145.0
diamond         144.0
diamond_c3      145.0
completa        145.0
Name: palabras, dtype: float64

Correlación de Spearman:

In [58]:
#para comparar dentro de la pregunta se le resta a cada valor la media de su pregunta
medidas = ["rouge_su4", "rouge_su4_primera", "bertscore", "rc_cobertura", "palabras"]
respuestas = summary[~summary["abstiene"]]
por_pregunta = respuestas[medidas] - respuestas.groupby("consulta")[medidas].transform("mean")
por_pregunta_y_configuracion = respuestas[medidas] - respuestas.groupby(["consulta", "configuracion"])[medidas].transform("mean")
pd.DataFrame({"todo junto": respuestas[medidas].corr(method="spearman")["palabras"],
              "dentro de la pregunta": por_pregunta.corr(method="spearman")["palabras"],
              "dentro de pregunta y configuracion": por_pregunta_y_configuracion.corr(method="spearman")["palabras"]}).drop("palabras").round(2)

,todo junto,dentro de la pregunta,dentro de pregunta y configuracion
rouge_su4,0.05,-0.11,-0.24
rouge_su4_primera,0.03,-0.11,-0.24
bertscore,-0.12,-0.18,-0.35
rc_cobertura,0.04,0.15,0.06


ROUGE y BERTScore bajan cuando la respuesta se alarga, mientras que la cobertura casi no se mueve, así que RAGChecker no parece castigar el largo. Hay que tener en cuenta que una respuesta más larga tiene más chances de tocar afirmaciones de la ideal, y que la cobertura la decide un LLM como juez, sin revisión humana.

# Costo que se tiene para cada configuración

In [59]:
import json

import pandas as pd

from ic import CONFIGURACIONES, SELECCION, NOMBRES

pd.set_option("display.width", 200)

In [60]:
conjuntos = ["factoid", "factoid_sin_oro", "list", "list_por_oro", "yesno", "summary"]
partes = []
for conjunto in conjuntos:
    parte = pd.read_csv(f"../gasto_por_pregunta_{conjunto}.csv")
    with open(f"../preguntas/muestra_{conjunto}.json") as archivo:
        muestra = json.load(archivo)
    parte["conjunto"] = conjunto
    parte["id_bioasq"] = [muestra[consulta]["id_bioasq"] for consulta in parte["consulta"]]
    partes.append(parte)
gasto = pd.concat(partes, ignore_index=True)
gasto["tokens"] = gasto["tokens_indice"] + gasto["tokens_consulta"]
gasto["segundos"] = gasto["segundos_seleccion"] + gasto["segundos_indice"] + gasto["segundos_consulta"]
#algunas preguntas estan en dos muestras: en la fila de todas cuentan una vez
sin_repetir = gasto.drop_duplicates(["id_bioasq", "configuracion"])
pd.concat([gasto.groupby("conjunto")["consulta"].nunique().reindex(conjuntos), pd.Series({"todas": sin_repetir["id_bioasq"].nunique()})])

factoid            111
factoid_sin_oro    111
list                51
list_por_oro        75
yesno              100
summary            100
todas              541
dtype: int64

## Tokens por pregunta

Media:


In [61]:
tokens = gasto.pivot_table(index="conjunto", columns="configuracion", values="tokens", aggfunc="mean").reindex(conjuntos)[CONFIGURACIONES]
tokens.loc["todas"] = sin_repetir.groupby("configuracion")["tokens"].mean()
tokens.round(0).to_csv("salidas/tokens_media.csv")
tokens.round(0)

configuracion,sin_grafo,grafo_entero,semillas,diamond,diamond_c3,completa
conjunto,,,,,,
factoid,194.0,66114.0,19852.0,32318.0,42495.0,60234.0
factoid_sin_oro,194.0,65585.0,30481.0,47661.0,70515.0,92184.0
list,221.0,65796.0,18933.0,33749.0,44415.0,59894.0
list_por_oro,245.0,65600.0,21962.0,37772.0,57772.0,78179.0
yesno,81.0,65641.0,38784.0,66117.0,96880.0,121649.0
summary,361.0,65623.0,26806.0,45244.0,64178.0,88915.0
todas,213.0,65738.0,26846.0,44683.0,64133.0,85329.0


Mediana:

In [62]:
tokens_mediana = gasto.pivot_table(index="conjunto", columns="configuracion", values="tokens", aggfunc="median").reindex(conjuntos)[CONFIGURACIONES]
tokens_mediana.loc["todas"] = sin_repetir.groupby("configuracion")["tokens"].median()
tokens_mediana.round(0).to_csv("salidas/tokens_mediana.csv")
tokens_mediana.round(0)

configuracion,sin_grafo,grafo_entero,semillas,diamond,diamond_c3,completa
conjunto,,,,,,
factoid,192.0,65543.0,7631.0,8233.0,8184.0,21518.0
factoid_sin_oro,194.0,65071.0,9695.0,24934.0,27440.0,45996.0
list,202.0,65322.0,6185.0,9060.0,9072.0,23432.0
list_por_oro,198.0,65091.0,8490.0,9313.0,9313.0,24004.0
yesno,80.0,65260.0,16949.0,49594.0,69993.0,86566.0
summary,358.0,65206.0,8589.0,25422.0,24610.0,52009.0
todas,195.0,65201.0,8835.0,23967.0,24460.0,41319.0


Índice y consulta por separado:

In [63]:
sin_repetir.groupby("configuracion")[["tokens_indice", "tokens_consulta"]].mean().reindex(CONFIGURACIONES).round(0)

,tokens_indice,tokens_consulta
configuracion,,
sin_grafo,0.0,213.0
grafo_entero,0.0,65738.0
semillas,23438.0,3408.0
diamond,41166.0,3517.0
diamond_c3,60035.0,4098.0
completa,80584.0,4746.0


## Segundos por pregunta

Los segundos dependen de la carga de la GPU. Media:

In [64]:
segundos = gasto.pivot_table(index="conjunto", columns="configuracion", values="segundos", aggfunc="mean").reindex(conjuntos)[CONFIGURACIONES]
segundos.loc["todas"] = sin_repetir.groupby("configuracion")["segundos"].mean()
segundos.round(1).to_csv("salidas/segundos_media.csv")
segundos.round(1)

configuracion,sin_grafo,grafo_entero,semillas,diamond,diamond_c3,completa
conjunto,,,,,,
factoid,0.3,27.3,53.0,69.7,84.5,112.9
factoid_sin_oro,0.3,26.0,71.6,98.9,127.8,159.8
list,0.4,26.4,53.0,74.1,89.4,112.8
list_por_oro,0.6,24.8,58.0,80.5,106.3,143.8
yesno,0.2,26.7,81.7,125.4,166.4,208.4
summary,1.8,28.3,67.2,96.3,117.0,157.0
todas,0.6,26.7,65.2,92.5,117.6,151.9


Mediana:

In [65]:
segundos_mediana = gasto.pivot_table(index="conjunto", columns="configuracion", values="segundos", aggfunc="median").reindex(conjuntos)[CONFIGURACIONES]
segundos_mediana.loc["todas"] = sin_repetir.groupby("configuracion")["segundos"].median()
segundos_mediana.round(1).to_csv("salidas/segundos_mediana.csv")
segundos_mediana.round(1)

configuracion,sin_grafo,grafo_entero,semillas,diamond,diamond_c3,completa
conjunto,,,,,,
factoid,0.2,26.1,38.4,35.2,35.4,52.2
factoid_sin_oro,0.2,24.1,44.5,60.0,65.8,103.3
list,0.2,24.3,39.5,40.1,39.1,56.6
list_por_oro,0.2,22.0,40.6,40.6,43.0,68.3
yesno,0.1,25.3,52.3,103.9,126.7,159.0
summary,1.7,25.7,43.8,64.8,65.2,101.5
todas,0.2,24.8,42.9,56.1,58.7,85.5


## El índice del grafo entero

Cuántas preguntas hacen falta para que convenga pagarlo: su costo sobre lo que cada selección gasta de más por pregunta contra la consulta al grafo entero. Vale para el grafo entero en nivel 0 y supone que cada pregunta paga su índice. NaN quiere decir que la selección gasta menos y no se amortiza nunca.

In [66]:
with open("../../grafo_entero/reportes250_map200_respuesta400_lotes5000_comunidades20_texto/indice.json") as archivo:
    indice = json.load(archivo)
tokens_del_indice = indice["tokens_entrada_indice"] + indice["tokens_salida_indice"]
segundos_del_indice = indice["segundos_total"]
pd.Series({"tokens": tokens_del_indice, "segundos": round(segundos_del_indice), "llamadas": indice["llamadas_indice"]})

tokens      65270854
segundos       81824
llamadas       16244
dtype: int64

En tokens:

In [67]:
de_mas = tokens[SELECCION].sub(tokens["grafo_entero"], axis=0)
equilibrio_tokens = (tokens_del_indice / de_mas).where(de_mas > 0).round(0)
equilibrio_tokens.to_csv("salidas/equilibrio_tokens.csv")
equilibrio_tokens

configuracion,semillas,diamond,diamond_c3,completa
conjunto,,,,
factoid,NaN,NaN,NaN,NaN
factoid_sin_oro,NaN,NaN,13239.0,2454.0
list,NaN,NaN,NaN,NaN
list_por_oro,NaN,NaN,NaN,5189.0
yesno,NaN,137251.0,2089.0,1165.0
summary,NaN,NaN,NaN,2802.0
todas,NaN,NaN,NaN,3332.0


En segundos:

In [68]:
de_mas = segundos[SELECCION].sub(segundos["grafo_entero"], axis=0)
equilibrio_segundos = (segundos_del_indice / de_mas).where(de_mas > 0).round(0)
equilibrio_segundos.to_csv("salidas/equilibrio_segundos.csv")
equilibrio_segundos

configuracion,semillas,diamond,diamond_c3,completa
conjunto,,,,
factoid,3181.0,1930.0,1430.0,956.0
factoid_sin_oro,1794.0,1122.0,803.0,612.0
list,3071.0,1714.0,1298.0,947.0
list_por_oro,2464.0,1469.0,1003.0,687.0
yesno,1487.0,829.0,586.0,450.0
summary,2104.0,1204.0,923.0,636.0
todas,2123.0,1243.0,900.0,653.0


## Con reuso de índices

El subgrafo depende solo de las entidades, así que preguntas con las mismas entidades pueden compartir índice. 

In [69]:
una_vez = sin_repetir.drop_duplicates(["clave", "configuracion"])
preguntas = sin_repetir.groupby("configuracion").size()
tokens_reuso = (una_vez.groupby("configuracion")["tokens_indice"].sum() + sin_repetir.groupby("configuracion")["tokens_consulta"].sum()) / preguntas
segundos_reuso = (una_vez.groupby("configuracion")[["segundos_seleccion", "segundos_indice"]].sum().sum(axis=1) + sin_repetir.groupby("configuracion")["segundos_consulta"].sum()) / preguntas
reuso = pd.DataFrame({"tokens": tokens_reuso, "segundos": segundos_reuso}).reindex(CONFIGURACIONES)
reuso["equilibrio_tokens"] = (tokens_del_indice / (reuso["tokens"] - reuso.loc["grafo_entero", "tokens"])).where(reuso["tokens"] > reuso.loc["grafo_entero", "tokens"])
reuso["equilibrio_segundos"] = (segundos_del_indice / (reuso["segundos"] - reuso.loc["grafo_entero", "segundos"])).where(reuso["segundos"] > reuso.loc["grafo_entero", "segundos"])
reuso = reuso.loc[SELECCION + ["grafo_entero"]].round({"tokens": 0, "segundos": 1, "equilibrio_tokens": 0, "equilibrio_segundos": 0})
reuso.to_csv("salidas/reuso.csv")
reuso

,tokens,segundos,equilibrio_tokens,equilibrio_segundos
configuracion,,,,
semillas,21561.0,55.3,NaN,2858.0
diamond,37183.0,78.4,NaN,1583.0
diamond_c3,50998.0,95.6,NaN,1187.0
completa,69067.0,126.7,19604.0,818.0
grafo_entero,65738.0,26.7,NaN,NaN


## Costo contra calidad

Tokens por pregunta (media) y medida principal de cada tarea, que escribe la primera sección. Del grafo entero se cuenta solo la consulta.

In [70]:
tabla = pd.DataFrame(index=[NOMBRES[configuracion] for configuracion in CONFIGURACIONES])
for tarea, medida in [("factoid", "laxa"), ("list", "F1"), ("yesno", "exactitud"), ("summary", "cobertura")]:
    principal = pd.read_csv(f"salidas/medida_principal_{tarea}.csv", index_col="configuracion")
    tabla[f"{tarea} tokens"] = tokens.loc[tarea, CONFIGURACIONES].round(0).astype(int).to_numpy()
    tabla[f"{tarea} {medida}"] = principal["media"]
tabla.to_csv("salidas/costo_calidad.csv")
tabla

,factoid tokens,factoid laxa,list tokens,list F1,yesno tokens,yesno exactitud,summary tokens,summary cobertura
modelo solo,194,0.613,221,0.426,81,0.847,361,0.488
grafo entero,66114,0.583,65796,0.462,65641,0.817,65623,0.446
solo semillas,19852,0.634,18933,0.536,38784,0.737,26806,0.342
DIAMOnD,32318,0.628,33749,0.552,66117,0.693,45244,0.361
DIAMOnD y C3,42495,0.613,44415,0.539,96880,0.723,64178,0.361
completa,60234,0.622,59894,0.555,121649,0.763,88915,0.347
